## Importation

In [30]:
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer , AutoModelForSequenceClassification
from tqdm.notebook import tqdm
import os
import numpy as np
import re
import html


In [9]:
! pip install ipywidgets

In [11]:
print(f" PyTorch version : {torch.__version__}")
print(f" Transformers version : {transformers.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Le matériel détecté pour les calculs IA est : {device}")

 PyTorch version : 2.5.1+cu121
 Transformers version : 4.57.6
 Le matériel détecté pour les calculs IA est : cuda


## Benchmarking

In [6]:
from transformers import AutoTokenizer


modeles_challengers = {
    " Le Classique (ProsusAI)": "ProsusAI/finbert",
    " L'Expert Corporate (Yiyang)": "yiyanghkust/finbert-tone",
    " Le Moderne (Ahmed)": "ahmedrachid/FinancialBERT-Sentiment-Analysis"
}


phrase_test = "Tesla earnings are skyrocketing today!"

print(f" PHRASE ORIGINALE : '{phrase_test}'")



for alias, nom_modele in modeles_challengers.items():
    print(f" Chargement du Tokenizer pour {alias}...")
    tokenizer = AutoTokenizer.from_pretrained(nom_modele)
    
   
    traduction = tokenizer(phrase_test, return_tensors="pt")
    
    
    tokens = tokenizer.convert_ids_to_tokens(traduction["input_ids"][0])
    input_ids = traduction["input_ids"][0].tolist()
    
    
    print(f"  Découpage (Tokens) : {tokens}")
    print(f" Input IDs (Chiffres) : {input_ids}")
    print("-" * 60, "\n")

 PHRASE ORIGINALE : 'Tesla earnings are skyrocketing today!'
 Chargement du Tokenizer pour  Le Classique (ProsusAI)...


c:\Users\semy4\OneDrive\Bureau\Fintech_project\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\semy4\.cache\huggingface\hub\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  Découpage (Tokens) : ['[CLS]', 'tesla', 'earnings', 'are', 'sky', '##rock', '##eti', '##ng', 'today', '!', '[SEP]']
 Input IDs (Chiffres) : [101, 26060, 16565, 2024, 3712, 16901, 20624, 3070, 2651, 999, 102]
------------------------------------------------------------ 

 Chargement du Tokenizer pour  L'Expert Corporate (Yiyang)...


c:\Users\semy4\OneDrive\Bureau\Fintech_project\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\semy4\.cache\huggingface\hub\models--yiyanghkust--finbert-tone. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  Découpage (Tokens) : ['[CLS]', 'tesla', 'earnings', 'are', 'sky', '##rock', '##etin', '##g', 'today', '!', '[SEP]']
 Input IDs (Chiffres) : [3, 22679, 149, 21, 7601, 12896, 13753, 847, 1163, 17293, 4]
------------------------------------------------------------ 

 Chargement du Tokenizer pour  Le Moderne (Ahmed)...


c:\Users\semy4\OneDrive\Bureau\Fintech_project\venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\semy4\.cache\huggingface\hub\models--ahmedrachid--FinancialBERT-Sentiment-Analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


  Découpage (Tokens) : ['[CLS]', '[UNK]', 'earnings', 'are', 'sky', '##rock', '##etin', '##g', 'today', '!', '[SEP]']
 Input IDs (Chiffres) : [3, 2, 149, 21, 7601, 12896, 13753, 847, 1163, 17293, 4]
------------------------------------------------------------ 



In [14]:


tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")

# On passe un lot (Batch) de 2 phrases de tailles très différentes
phrases = [
    "Tesla earnings are skyrocketing today!", 
    "Buy Tesla."
]

# padding=True est le mot magique qui aligne les tailles
traduction = tokenizer(phrases, padding=True, return_tensors="pt")

print("🔢 Input IDs (Les numéros des mots) :")
print("Phrase 1 :", traduction["input_ids"][0].tolist())
print("Phrase 2 :", traduction["input_ids"][1].tolist(), "\n")

print("🔦 Attention Mask (Les 1 et les 0) :")
print("Phrase 1 :", traduction["attention_mask"][0].tolist())
print("Phrase 2 :", traduction["attention_mask"][1].tolist())

🔢 Input IDs (Les numéros des mots) :
Phrase 1 : [3, 22679, 149, 21, 7601, 12896, 13753, 847, 1163, 17293, 4]
Phrase 2 : [3, 500, 22679, 48, 4, 0, 0, 0, 0, 0, 0] 

🔦 Attention Mask (Les 1 et les 0) :
Phrase 1 : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Phrase 2 : [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]


In [16]:

nom_modele = "ProsusAI/finbert"
print(f" Chargement de l'IA {nom_modele}...")

tokenizer = AutoTokenizer.from_pretrained(nom_modele)


model = AutoModelForSequenceClassification.from_pretrained(nom_modele, use_safetensors=True)
print(" Modèle prêt !\n")


phrases = [
    "Tesla earnings are skyrocketing today!",       # Devrait être Positif
    "The company is facing a massive bankruptcy.",  # Devrait être Négatif
    "Apple releases its new quarterly report."      # Devrait être Neutre
]

# 3. Tokenization 
inputs = tokenizer(phrases, padding=True, truncation=True, return_tensors="pt")

# 4. L'Inférence (Le calcul de prédiction)
print(" Calcul des prédictions en cours...")
with torch.no_grad(): 
    outputs = model(**inputs)

# 5. La transformation en Probabilités avec Softmax
probabilites = torch.nn.functional.softmax(outputs.logits, dim=-1)

print("\n Probabilités finales (Ordre de FinBERT : [Positif, Négatif, Neutre]) :")
for i, phrase in enumerate(phrases):
    probs = probabilites[i].numpy()
    print(f"\nPhrase : '{phrase}'")
    print(f"   🟢 Positif : {probs[0]*100:.1f}%")
    print(f"   🔴 Négatif : {probs[1]*100:.1f}%")
    print(f"   ⚪ Neutre  : {probs[2]*100:.1f}%")

 Chargement de l'IA ProsusAI/finbert...
 Modèle prêt !

 Calcul des prédictions en cours...

 Probabilités finales (Ordre de FinBERT : [Positif, Négatif, Neutre]) :

Phrase : 'Tesla earnings are skyrocketing today!'
   🟢 Positif : 29.1%
   🔴 Négatif : 61.3%
   ⚪ Neutre  : 9.6%

Phrase : 'The company is facing a massive bankruptcy.'
   🟢 Positif : 1.0%
   🔴 Négatif : 90.9%
   ⚪ Neutre  : 8.1%

Phrase : 'Apple releases its new quarterly report.'
   🟢 Positif : 2.3%
   🔴 Négatif : 27.2%
   ⚪ Neutre  : 70.5%


In [18]:
! pip install hf_xet

In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nom_modele = "yiyanghkust/finbert-tone"
print(f" Chargement de l'IA {nom_modele}...")

tokenizer = AutoTokenizer.from_pretrained(nom_modele)


model = AutoModelForSequenceClassification.from_pretrained(nom_modele, use_safetensors=True)
print("Modèle prêt !\n")

phrases = [
    "Tesla earnings are skyrocketing today!",       
    "The company is facing a massive bankruptcy.",  
    "Apple releases its new quarterly report."      
]

inputs = tokenizer(phrases, padding=True, truncation=True, return_tensors="pt")

print("  Calcul des prédictions en cours...")
with torch.no_grad(): 
    outputs = model(**inputs)

probabilites = torch.nn.functional.softmax(outputs.logits, dim=-1)

dictionnaire_labels = model.config.id2label

print("Probabilités finales :")
for i, phrase in enumerate(phrases):
    probs = probabilites[i].numpy()
    print(f"\nPhrase : '{phrase}'")
    
    for id_label, nom_label in dictionnaire_labels.items():
        score = probs[id_label] * 100
        if nom_label.lower() == "positive":
            icone = "🟢"
        elif nom_label.lower() == "negative":
            icone = "🔴"
        else:
            icone = "⚪"
            
        print(f"   {icone} {nom_label.capitalize()} : {score:.1f}%")

⏳ Chargement de l'IA yiyanghkust/finbert-tone...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


✅ Modèle prêt !

⚙️  Calcul des prédictions en cours...
📊 Probabilités finales :

Phrase : 'Tesla earnings are skyrocketing today!'
   ⚪ Neutral : 100.0%
   🟢 Positive : 0.0%
   🔴 Negative : 0.0%

Phrase : 'The company is facing a massive bankruptcy.'
   ⚪ Neutral : 0.0%
   🟢 Positive : 0.0%
   🔴 Negative : 100.0%

Phrase : 'Apple releases its new quarterly report.'
   ⚪ Neutral : 100.0%
   🟢 Positive : 0.0%
   🔴 Negative : 0.0%


---
### 🏆 Conclusion du Benchmarking et Choix du Modèle

Avant de lancer le traitement massif de notre dataset, nous avons procédé à un **Benchmarking** (test comparatif) de plusieurs architectures d'Intelligence Artificielle spécialisées dans la finance.

L'objectif était d'observer comment différents modèles interprètent un vocabulaire boursier piégeur.

### 📊 Les Candidats et Résultats

Nous avons testé la phrase piège : *"Tesla earnings are skyrocketing today!"* (Bénéfices qui explosent).

1. **`ProsusAI/finbert` (Le standard de la presse)**
   * **Comportement :** Produit des probabilités nuancées (ex: 61% / 29% / 9%).
   * **Le Biais :** A classé la phrase en **Négatif**. Étant entraîné sur des articles de presse, il associe statistiquement le mot *"skyrocketing"* à des événements négatifs (inflation, dettes, coûts). Il se laisse tromper par la forme.

2. **`yiyanghkust/finbert-tone` (L'Expert Corporate)**
   * **Comportement :** Produit des probabilités extrêmement tranchées, souvent binaires (**100% ou 0%**).
   * **La Force :** A classé la phrase en **Neutre**. Étant entraîné sur des documents comptables officiels (10-K), il est totalement immunisé contre le sensationnalisme ou la "hype". Si l'information n'utilise pas le jargon strict de la hausse des revenus, il refuse de s'emballer. En revanche, il sanctionne impitoyablement les vrais mots de crise (ex: *bankruptcy* = 100% Négatif).

### ⚖️ Décision Architecturale

> **✅ Modèle sélectionné : `yiyanghkust/finbert-tone`**

Nous choisissons l'Expert Corporate. Sa rigueur institutionnelle, sa capacité à ne pas se laisser tromper par le vocabulaire exagéré des réseaux sociaux/articles, et ses prédictions ultra-confiantes nous permettront d'obtenir un signal de trading beaucoup plus net et moins bruité.

---
**➡️ Prochaine étape :** Mise en production et Inférence par Lots (Batch Processing) sur l'ensemble du dataset en utilisant l'accélération GPU (RTX 3075).

## ⚙️ Étape 2 : Préparation des Données & Tokenization Industrielle

Cette section prépare notre dataset réel pour l'envoyer au modèle de manière sécurisée et optimisée. 

### Objectifs :
1. **Sécurisation des types :** Remplissage des valeurs manquantes (`NaN`) pour éviter les plantages lors de la lecture.
2. **Définition de la longueur maximale (`max_length=128`) :** Limite la taille des vecteurs pour économiser la mémoire de la RTX 3075.
3. **Validation de la structure :** Génération des premiers tenseurs (`input_ids` et `attention_mask`) sur un échantillon pour valider la forme de notre matrice de calcul.

In [32]:
df=pd.read_csv(r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\MASTER_DATASET_BRUT_MAX_processed.csv", sep=",")

In [33]:
df.head()

,Date,Source,Ticker,Titre
0,2026-07-12,Yahoo,MSFT,"Could SpaceX's Starmind Make Amazon's, Microso..."
1,2026-07-12,Yahoo,TSLA,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...
2,2026-07-12,Yahoo,AMZN,Synchrony (SYF) Announces Executive Leadership...
3,2026-07-12,CNBC,AMZN,Top Wall Street analysts are confident about t...
4,2026-07-12,Yahoo,AMZN,The U.S. Economy Is Addicted to AI Spending. W...


In [34]:

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

df_2023_2026 = df[(df['Date'].dt.year >= 2023) & (df['Date'].dt.year <= 2026)].copy()


chemin_sauvegarde = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\MASTER_DATASET_2023_2026.csv"
df_2023_2026.to_csv(chemin_sauvegarde, index=False)

print(f" Lignes avant filtrage : {len(df)}")
print(f"Lignes conservées (2023-2026) : {len(df_2023_2026)}")
display(df_2023_2026.head())

 Lignes avant filtrage : 3897566
Lignes conservées (2023-2026) : 33059


,Date,Source,Ticker,Titre
0,2026-07-12,Yahoo,MSFT,"Could SpaceX's Starmind Make Amazon's, Microso..."
1,2026-07-12,Yahoo,TSLA,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...
2,2026-07-12,Yahoo,AMZN,Synchrony (SYF) Announces Executive Leadership...
3,2026-07-12,CNBC,AMZN,Top Wall Street analysts are confident about t...
4,2026-07-12,Yahoo,AMZN,The U.S. Economy Is Addicted to AI Spending. W...


In [35]:
df_2023_2026.isnull().sum()

Date      0
Source    0
Ticker    0
Titre     0
dtype: int64

In [36]:


def nettoyer_texte_finbert(texte):
    
    if not isinstance(texte, str):
        return ""
    
    
    texte = html.unescape(texte)
    
    
    texte = re.sub(r'http\S+|www\S+|https\S+', '', texte, flags=re.MULTILINE)
    
    
    texte = re.sub(r'\@\w+', '', texte)
    
    
    texte = re.sub(r'\s+', ' ', texte).strip()
    
    return texte


print(" Lancement du nettoyage...")
df_2023_2026['Texte_Nettoye'] = df_2023_2026['Titre'].apply(nettoyer_texte_finbert)

print(" Nettoyage terminé !")
display(df_2023_2026[['Titre', 'Texte_Nettoye']].head())

 Lancement du nettoyage...
 Nettoyage terminé !


,Titre,Texte_Nettoye
0,"Could SpaceX's Starmind Make Amazon's, Microso...","Could SpaceX's Starmind Make Amazon's, Microso..."
1,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...
2,Synchrony (SYF) Announces Executive Leadership...,Synchrony (SYF) Announces Executive Leadership...
3,Top Wall Street analysts are confident about t...,Top Wall Street analysts are confident about t...
4,The U.S. Economy Is Addicted to AI Spending. W...,The U.S. Economy Is Addicted to AI Spending. W...


In [37]:
df_2023_2026.isnull().sum()

Date             0
Source           0
Ticker           0
Titre            0
Texte_Nettoye    0
dtype: int64

In [38]:
nom_modele = "yiyanghkust/finbert-tone"
print(f" Chargement du dictionnaire : {nom_modele}...")
tokenizer = AutoTokenizer.from_pretrained(nom_modele)   
colonne_texte = 'Titre'
textes_test = df_2023_2026[colonne_texte].head(5).tolist()


inputs = tokenizer(
    textes_test, 
    padding=True, 
    truncation=True, 
    max_length=128, 
    return_tensors="pt"
)

print("\n Forme finale de la matrice mathématique (Tenseurs) :")
print(f"Nombre de phrases traitées : {inputs['input_ids'].shape[0]}")
print(f"Longueur du vecteur (avec Padding) : {inputs['input_ids'].shape[1]} tokens")

 Chargement du dictionnaire : yiyanghkust/finbert-tone...

 Forme finale de la matrice mathématique (Tenseurs) :
Nombre de phrases traitées : 5
Longueur du vecteur (avec Padding) : 92 tokens


## ⚙️ Étape 3 : Inférence par Lots (Batch Processing) & Accélération GPU

Cette étape constitue le cœur applicatif de notre projet. Nous allons faire passer l'intégralité du dataset nettoyé à travers l'Expert Corporate (`yiyanghkust/finbert-tone`).

### Stratégie technique :
1. **Accélération Matérielle :** Transfert automatique du modèle sur la VRAM de notre carte graphique (CUDA / RTX 3075).
2. **Découpage en Batchs (Taille = 64) :** Au lieu de traiter l'ensemble du dataset d'un coup, nous envoyons les données par groupes de 64 lignes pour préserver la RAM.
3. **Désactivation des Gradients (`torch.no_grad()`) :** Désactive l'apprentissage du modèle pour doubler la vitesse de calcul et économiser l'énergie de la carte graphique.
4. **Exportation Sécurisée :** Sauvegarde des résultats intermédiaires pour ne pas perdre les calculs en cas de coupure de courant.

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Moteur de calcul activé : {device.type.upper()} (RTX 3075 détectée !)")

 Moteur de calcul activé : CUDA (RTX 3075 détectée !)


In [40]:
nom_modele = "yiyanghkust/finbert-tone"
print(f" Chargement de l'IA {nom_modele}...")
tokenizer = AutoTokenizer.from_pretrained(nom_modele)

 Chargement de l'IA yiyanghkust/finbert-tone...


In [41]:
model = AutoModelForSequenceClassification.from_pretrained(nom_modele, use_safetensors=True).to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30873, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [43]:
df_2023_2026.dtypes

Date             datetime64[ns]
Source                   object
Ticker                   object
Titre                    object
Texte_Nettoye            object
dtype: object

In [44]:
df_2023_2026.info()

<class 'pandas.core.frame.DataFrame'>
Index: 33059 entries, 0 to 33058
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           33059 non-null  datetime64[ns]
 1   Source         33059 non-null  object        
 2   Ticker         33059 non-null  object        
 3   Titre          33059 non-null  object        
 4   Texte_Nettoye  33059 non-null  object        
dtypes: datetime64[ns](1), object(4)
memory usage: 1.5+ MB


In [45]:
df_2023_2026["Ticker"].unique()

array(['MSFT', 'TSLA', 'AMZN', 'JPM', 'GOOGL', 'META', 'AAPL', 'BRK-B',
       'UNH'], dtype=object)

In [46]:
df_2023_2026.head()

,Date,Source,Ticker,Titre,Texte_Nettoye
0,2026-07-12,Yahoo,MSFT,"Could SpaceX's Starmind Make Amazon's, Microso...","Could SpaceX's Starmind Make Amazon's, Microso..."
1,2026-07-12,Yahoo,TSLA,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...
2,2026-07-12,Yahoo,AMZN,Synchrony (SYF) Announces Executive Leadership...,Synchrony (SYF) Announces Executive Leadership...
3,2026-07-12,CNBC,AMZN,Top Wall Street analysts are confident about t...,Top Wall Street analysts are confident about t...
4,2026-07-12,Yahoo,AMZN,The U.S. Economy Is Addicted to AI Spending. W...,The U.S. Economy Is Addicted to AI Spending. W...


In [47]:
BATCH_SIZE = 64 
colonne_texte = 'Texte_Nettoye'
phrases = df_2023_2026[colonne_texte].tolist()

In [48]:
df_2023_2026.isnull().sum()

Date             0
Source           0
Ticker           0
Titre            0
Texte_Nettoye    0
dtype: int64

In [49]:
labels_finaux = []
scores_positifs, scores_negatifs, scores_neutres = [], [], []

In [50]:
label_names = [model.config.id2label[i] for i in range(3)]

In [51]:

print(f"\n⚡ Lancement de l'analyse sur {len(df_2023_2026)} textes...")
with torch.no_grad():
    for i in tqdm(range(0, len(phrases), BATCH_SIZE), desc="Analyse FinBERT"):
        batch_phrases = phrases[i:i + BATCH_SIZE]
        
        inputs = tokenizer(
            batch_phrases, 
            padding=True, 
            truncation=True, 
            max_length=128, 
            return_tensors="pt"
        ).to(device)
        
        # Prédiction
        outputs = model(**inputs)
        probabilites = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
        
        # Extraction des scores
        for probs in probabilites:
            scores = {model.config.id2label[idx]: val for idx, val in enumerate(probs)}
            
            scores_positifs.append(scores.get('Positive', 0.0))
            scores_negatifs.append(scores.get('Negative', 0.0))
            scores_neutres.append(scores.get('Neutral', 0.0))
            
            labels_finaux.append(max(scores, key=scores.get))


⚡ Lancement de l'analyse sur 33059 textes...


Analyse FinBERT:   0%|          | 0/517 [00:00<?, ?it/s]

In [54]:

df_2023_2026['Sentiment_FinBERT'] = labels_finaux
df_2023_2026['FinBERT_Positive'] = scores_positifs
df_2023_2026['FinBERT_Negative'] = scores_negatifs
df_2023_2026['FinBERT_Neutral'] = scores_neutres

# 6. Sauvegarde intelligente
dossier_sauvegarde = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed"
# On donne un nom clair pour retrouver cet échantillon facilement
fichier_sortie = os.path.join(dossier_sauvegarde, "data_2023_2026_PREDICTED.csv")
df_2023_2026.to_csv(fichier_sortie, index=False)

print(f"\n💾 Succès ! Le fichier avec les scores de sentiment a été enregistré ici :")
print(f"👉 {fichier_sortie}")


💾 Succès ! Le fichier avec les scores de sentiment a été enregistré ici :
👉 C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\data_2023_2026_PREDICTED.csv


In [56]:
df_sen=pd.read_csv(r"C:\\Users\\semy4\\OneDrive\\Bureau\\Fintech_project\\data\\processed\\data_2023_2026_PREDICTED.csv")

In [57]:
df_sen

,Date,Source,Ticker,Titre,Texte_Nettoye,Sentiment_FinBERT,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral
0,2026-07-12,Yahoo,MSFT,"Could SpaceX's Starmind Make Amazon's, Microso...","Could SpaceX's Starmind Make Amazon's, Microso...",Neutral,4.892208e-03,3.011416e-01,6.939662e-01
1,2026-07-12,Yahoo,TSLA,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...,Neutral,5.112568e-04,6.916437e-03,9.925724e-01
2,2026-07-12,Yahoo,AMZN,Synchrony (SYF) Announces Executive Leadership...,Synchrony (SYF) Announces Executive Leadership...,Positive,9.908533e-01,1.471132e-05,9.132064e-03
3,2026-07-12,CNBC,AMZN,Top Wall Street analysts are confident about t...,Top Wall Street analysts are confident about t...,Positive,9.999995e-01,8.672569e-08,4.014215e-07
4,2026-07-12,Yahoo,AMZN,The U.S. Economy Is Addicted to AI Spending. W...,The U.S. Economy Is Addicted to AI Spending. W...,Negative,2.526199e-05,9.997814e-01,1.932713e-04
...,...,...,...,...,...,...,...,...,...
33054,2023-01-02,The Motley Fool,AAPL,Did Buffett Beat the Market in 2022? Here's Ho...,Did Buffett Beat the Market in 2022? Here's Ho...,Neutral,1.908794e-03,1.998557e-04,9.978914e-01
33055,2023-01-02,The Motley Fool,BRK-B,Did Buffett Beat the Market in 2022? Here's Ho...,Did Buffett Beat the Market in 2022? Here's Ho...,Neutral,1.908794e-03,1.998557e-04,9.978914e-01
33056,2023-01-02,Investor's Business Daily,TSLA,Tesla Stock Vs. BYD Stock: TSLA Ends Worst Yea...,Tesla Stock Vs. BYD Stock: TSLA Ends Worst Yea...,Negative,2.984695e-07,9.999992e-01,5.242631e-07
33057,2023-01-01,The Motley Fool,META,"This Company's Sales Soared 2,250% in 9 Years,...","This Company's Sales Soared 2,250% in 9 Years,...",Positive,9.999429e-01,8.521694e-06,4.858464e-05


## 🧩 Étape 3.1 : Agrégation des Scores (Par Jour et par Actif)

Pour pouvoir croiser nos milliers de textes quotidiens avec l'unique prix de clôture de la Bourse, nous devons d'abord synthétiser l'information. 
Conformément au cahier des charges, nous réalisons ici une **agrégation par jour et par actif**. 

**Objectif de l'agrégation :**
* Regrouper toutes les prédictions d'une même journée pour une même entreprise.
* Calculer la **moyenne** des probabilités (Positif, Négatif, Neutre).
* **Compter** le nombre total de messages pour obtenir notre premier indicateur de liquidité.

In [65]:
df_filtre=pd.read_csv(r"C:\\Users\\semy4\\OneDrive\\Bureau\\Fintech_project\\data\\processed\\data_2023_2026_PREDICTED.csv")

In [66]:
df_filtre

,Date,Source,Ticker,Titre,Texte_Nettoye,Sentiment_FinBERT,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral
0,2026-07-12,Yahoo,MSFT,"Could SpaceX's Starmind Make Amazon's, Microso...","Could SpaceX's Starmind Make Amazon's, Microso...",Neutral,4.892208e-03,3.011416e-01,6.939662e-01
1,2026-07-12,Yahoo,TSLA,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...,Elon Musk Says Tesla Robot Will Be ‘Biggest Pr...,Neutral,5.112568e-04,6.916437e-03,9.925724e-01
2,2026-07-12,Yahoo,AMZN,Synchrony (SYF) Announces Executive Leadership...,Synchrony (SYF) Announces Executive Leadership...,Positive,9.908533e-01,1.471132e-05,9.132064e-03
3,2026-07-12,CNBC,AMZN,Top Wall Street analysts are confident about t...,Top Wall Street analysts are confident about t...,Positive,9.999995e-01,8.672569e-08,4.014215e-07
4,2026-07-12,Yahoo,AMZN,The U.S. Economy Is Addicted to AI Spending. W...,The U.S. Economy Is Addicted to AI Spending. W...,Negative,2.526199e-05,9.997814e-01,1.932713e-04
...,...,...,...,...,...,...,...,...,...
33054,2023-01-02,The Motley Fool,AAPL,Did Buffett Beat the Market in 2022? Here's Ho...,Did Buffett Beat the Market in 2022? Here's Ho...,Neutral,1.908794e-03,1.998557e-04,9.978914e-01
33055,2023-01-02,The Motley Fool,BRK-B,Did Buffett Beat the Market in 2022? Here's Ho...,Did Buffett Beat the Market in 2022? Here's Ho...,Neutral,1.908794e-03,1.998557e-04,9.978914e-01
33056,2023-01-02,Investor's Business Daily,TSLA,Tesla Stock Vs. BYD Stock: TSLA Ends Worst Yea...,Tesla Stock Vs. BYD Stock: TSLA Ends Worst Yea...,Negative,2.984695e-07,9.999992e-01,5.242631e-07
33057,2023-01-01,The Motley Fool,META,"This Company's Sales Soared 2,250% in 9 Years,...","This Company's Sales Soared 2,250% in 9 Years,...",Positive,9.999429e-01,8.521694e-06,4.858464e-05


In [67]:
df_agrege = df_filtre.groupby(['Date', 'Ticker']).agg({
    'FinBERT_Positive': 'mean',   
    'FinBERT_Negative': 'mean',   
    'FinBERT_Neutral': 'mean',    
    'Texte_Nettoye': 'count' # On compte le nombre exact de textes publiés     
}).reset_index()

In [68]:
df_agrege

,Date,Ticker,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral,Texte_Nettoye
0,2023-01-01,BRK-B,9.986042e-01,0.000014,1.381718e-03,1
1,2023-01-01,META,9.999429e-01,0.000009,4.858464e-05,1
2,2023-01-02,AAPL,1.908794e-03,0.000200,9.978914e-01,1
3,2023-01-02,BRK-B,1.908794e-03,0.000200,9.978914e-01,1
4,2023-01-02,TSLA,2.984695e-07,0.999999,5.242631e-07,1
...,...,...,...,...,...,...
4904,2026-07-12,JPM,7.617720e-01,0.000018,2.382103e-01,4
4905,2026-07-12,META,4.320075e-01,0.360332,2.076605e-01,17
4906,2026-07-12,MSFT,2.831651e-01,0.295231,4.216034e-01,14
4907,2026-07-12,TSLA,2.849473e-01,0.330129,3.849236e-01,10


In [69]:
df_agrege.rename(columns={'Texte_Nettoye': 'Volume_Messages'}, inplace=True)

print("Agrégation réussie ! Tes données ont été compressées en un résumé par action :")
display(df_agrege)

Agrégation réussie ! Tes données ont été compressées en un résumé par action :


,Date,Ticker,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral,Volume_Messages
0,2023-01-01,BRK-B,9.986042e-01,0.000014,1.381718e-03,1
1,2023-01-01,META,9.999429e-01,0.000009,4.858464e-05,1
2,2023-01-02,AAPL,1.908794e-03,0.000200,9.978914e-01,1
3,2023-01-02,BRK-B,1.908794e-03,0.000200,9.978914e-01,1
4,2023-01-02,TSLA,2.984695e-07,0.999999,5.242631e-07,1
...,...,...,...,...,...,...
4904,2026-07-12,JPM,7.617720e-01,0.000018,2.382103e-01,4
4905,2026-07-12,META,4.320075e-01,0.360332,2.076605e-01,17
4906,2026-07-12,MSFT,2.831651e-01,0.295231,4.216034e-01,14
4907,2026-07-12,TSLA,2.849473e-01,0.330129,3.849236e-01,10


## 📈 Étape 3.2 : Création des Indicateurs de Sentiment (Feature Engineering)

Maintenant que nos données sont agrégées, nous passons à la "Création d'indicateurs de sentiment exploitables". L'objectif est de transformer nos probabilités brutes en véritables signaux de trading quantitatifs.

**Les 4 indicateurs calculés :**

*   **Le Volume de Messages (`Volume_Messages`) :** Mesure la "liquidité de l'information". Il garantit la fiabilité statistique du signal (déjà calculé lors de l'agrégation).
*   **Le Score de Sentiment Net (`Score_Net`) :** Mesure la force directionnelle globale du marché.
    $$Score\_Net = Moyenne(Positif) - Moyenne(Negatif)$$
*   **L'Indice d'Optimisme (`Bullishness_Index`) :** Isole la conviction forte en ignorant les indécis (le bruit neutre).
    $$Bullishness = \frac{Positif}{Positif + Negatif}$$
*   **L'Indice de Polarité (`Polarite_Index`) :** Mesure le niveau de désaccord (guerre entre acheteurs et vendeurs) pour anticiper les tempêtes et la volatilité.
    $$Polarite = 1 - Neutre$$

In [70]:
df_agrege['Score_Net'] = df_agrege['FinBERT_Positive'] - df_agrege['FinBERT_Negative']

df_agrege['Bullishness_Index'] = df_agrege['FinBERT_Positive'] / (df_agrege['FinBERT_Positive'] + df_agrege['FinBERT_Negative'] + 1e-9)


df_agrege['Polarite_Index'] = 1 - df_agrege['FinBERT_Neutral']


colonnes_a_arrondir = [
    'FinBERT_Positive', 'FinBERT_Negative', 'FinBERT_Neutral', 
    'Score_Net', 'Bullishness_Index', 'Polarite_Index'
]
df_agrege[colonnes_a_arrondir] = df_agrege[colonnes_a_arrondir].round(4)


chemin_indicateurs = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\data_2023_2026_INDICATEURS.csv"
df_agrege.to_csv(chemin_indicateurs, index=False)

print(f" Étape 3 officiellement terminée ! Les indicateurs sont sauvegardés dans :\n{chemin_indicateurs}\n")


display(df_agrege)

 Étape 3 officiellement terminée ! Les indicateurs sont sauvegardés dans :
C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed\data_2023_2026_INDICATEURS.csv



,Date,Ticker,FinBERT_Positive,FinBERT_Negative,FinBERT_Neutral,Volume_Messages,Score_Net,Bullishness_Index,Polarite_Index
0,2023-01-01,BRK-B,0.9986,0.0000,0.0014,1,0.9986,1.0000,0.9986
1,2023-01-01,META,0.9999,0.0000,0.0000,1,0.9999,1.0000,1.0000
2,2023-01-02,AAPL,0.0019,0.0002,0.9979,1,0.0017,0.9052,0.0021
3,2023-01-02,BRK-B,0.0019,0.0002,0.9979,1,0.0017,0.9052,0.0021
4,2023-01-02,TSLA,0.0000,1.0000,0.0000,1,-1.0000,0.0000,1.0000
...,...,...,...,...,...,...,...,...,...
4904,2026-07-12,JPM,0.7618,0.0000,0.2382,4,0.7618,1.0000,0.7618
4905,2026-07-12,META,0.4320,0.3603,0.2077,17,0.0717,0.5452,0.7923
4906,2026-07-12,MSFT,0.2832,0.2952,0.4216,14,-0.0121,0.4896,0.5784
4907,2026-07-12,TSLA,0.2849,0.3301,0.3849,10,-0.0452,0.4633,0.6151
